# Modèle de langage et génération de séquence — Transformeurs

Dans ces travaux pratiques, nous allons étudier les différentes manières de décoder du texte à partir d'un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage : traduction, question/réponse, génération…

Nous prendrons l'exemple de la génération de texte avec GPT-2, un transformeur de type *décodeur seul* : les grands modèles de langage actuels (GPT-4, Llama, Mistral…) reposent sur le même principe et se décodent avec les mêmes méthodes.

Nous partirons de la méthode la plus simple pour progresser jusqu'aux méthodes utilisées en pratique. Cela vous permettra de bien comprendre le rôle de la softmax en sortie des modèles de langage.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de Hugging Face peut servir de ressource complémentaire à ce notebook.

**Attention** : ces modèles ont été entraînés sur des textes du web, sans filtrage. Les textes générés, surtout par échantillonnage, peuvent être incohérents, faux ou choquants.

## Installation

La librairie [`transformers`](https://github.com/huggingface/transformers) de [Hugging Face](https://huggingface.co/) met à disposition un grand nombre de modèles de la famille des transformeurs. Elle est préinstallée sur Colab ; la cellule suivante s'assure simplement qu'elle est présente.

In [ ]:
!pip install -q transformers

## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles pré-entraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste du TP.

In [ ]:
lang = "fr"
# lang = "en"

## Import d'un modèle entraîné

En plus de mettre à disposition le code source de nombreuses architectures de transformeurs, la librairie `transformers` permet de récupérer des [modèles entraînés](https://huggingface.co/models) depuis le Hub Hugging Face.

Pour ce TP, nous allons utiliser l'architecture GPT-2 (2019). Il suffit d'utiliser la méthode `from_pretrained` avec le nom du modèle que l'on souhaite récupérer. On récupère le tokeniseur lié au modèle de la même manière.

Les modèles PyTorch sont des `nn.Module` : on les déplace sur le GPU avec `.to(device)`, et on les passe en mode évaluation avec `.eval()` (le dropout est alors désactivé).

In [ ]:
import functools

import torch
import tqdm.auto
import transformers

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs sur : {device}")

pretraining_name = "antoiloui/belgpt2" if lang == "fr" else "gpt2"

model = transformers.AutoModelForCausalLM.from_pretrained(pretraining_name)
model = model.to(device).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(pretraining_name)

## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques. Pour cela, on peut afficher sa configuration et le modèle lui-même (un `nn.Module` s'affiche avec toutes ses couches).

- *Combien le modèle a-t-il de couches de transformeur ?*
- *Quels sont ses mécanismes de régularisation ?*
- *De stabilisation de l'apprentissage ?*
- *Combien de paramètres possède-t-il ?*
- *Nous aurons besoin plus tard de l'index du symbole spécial utilisé pendant l'entraînement pour délimiter les début et fin de texte. Quel est-il ?*

In [ ]:
print(model.config)
print(model)

*Votre réponse à compléter ici.*
-
-
-
-
-

### Solution

In [ ]:
n_parameters = sum(p.numel() for p in model.parameters())
print(f"{n_parameters / 1e6:.0f} millions de paramètres")
print(f"Couches : {model.config.n_layer}")
print(f"Début/fin de texte : {model.config.bos_token_id} / {model.config.eos_token_id}")

- Il possède 12 couches de transformeur (`n_layer`, ou les 12 `GPT2Block` affichés).
- Son principal mécanisme de régularisation est le dropout (`attn_pdrop`, `resid_pdrop`, `embd_pdrop`).
- Ses mécanismes de stabilisation de l'apprentissage sont la normalisation de couche (`LayerNorm`, `ln_1`, `ln_2`) et les connexions résiduelles.
- Il a environ 124 millions de paramètres.
- On lit l'index des symboles de début et de fin de texte dans les clés `bos_token_id` et `eos_token_id` de la config.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on procède de manière itérative : on génère les mots un par un, en commençant par un texte vide ou par une phrase que l'on souhaite compléter.

Pour générer un mot, on donne en entrée du réseau les mots générés jusque-là (transformés en indices d'embedding). Le réseau produit alors un score par mot du vocabulaire. On choisit l'un de ces mots, on l'ajoute à ce qui a déjà été généré, puis on recommence pour générer le mot suivant.

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$$

Tout l'enjeu du décodage est de trouver le texte qui maximise ce produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approcher.

C'est le **masque causal** du transformeur qui rend cette décomposition possible : pendant l'entraînement, chaque position ne voit que les positions précédentes.

## Tokenisation

Le tokeniseur récupéré en même temps que le modèle transforme une phrase en indices de vocabulaire (des *sous-mots*) que le modèle comprend.

Nous utiliserons dans la suite du TP les fonctions `encode` et `decode` suivantes. `encode` ajoute le symbole de début de texte, et renvoie un tenseur de forme `(1, nombre de tokens)` sur le bon device.

In [ ]:
def encode(sentence: str) -> torch.Tensor:
  tokens = tokenizer.encode(sentence, add_special_tokens=False,
                            return_tensors="pt")
  bos = torch.tensor([[model.config.bos_token_id]])
  return torch.cat([bos, tokens], dim=1).to(device)


def decode(tokens: torch.Tensor) -> str:
  return tokenizer.decode(tokens.squeeze(0)[1:])


example = encode("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")
print(example)
print(tokenizer.convert_ids_to_tokens(example[0]))

## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain mot. Étudions son comportement.

Comme tout module PyTorch, on calcule la passe forward avec `model(input_ids)`. Le résultat est un objet dont les attributs principaux sont :

- `logits` : les scores du modèle, avant softmax ;
- `past_key_values` : un cache des clés et valeurs d'attention déjà calculées.

*Étudiez les sorties du modèle ([documentation](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)) en l'appliquant à diverses séquences, par exemple :*

- *«&nbsp;&nbsp;» (le texte vide)*
- *«&nbsp;Je pense, donc je suis&nbsp;» ou «&nbsp;I think, therefore I am&nbsp;» en fonction de la langue du modèle*

*Quelle est la forme des `logits` ? À quoi correspond chaque dimension ?*

*Pensez à désactiver le calcul des gradients avec `torch.no_grad()` : nous n'entraînons pas le modèle.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def describe_shapes(input_string: str) -> None:
  with torch.no_grad():
    output = model(encode(input_string))
  print("-" * 80)
  print(f"Forme des logits pour l'input '{input_string}' : "
        f"{tuple(output.logits.shape)}")
  print(f"Type du cache : {type(output.past_key_values).__name__}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")

Les logits ont la forme `(batch, nombre de tokens, taille du vocabulaire)` : pour chaque position de l'entrée, le modèle donne un score à chaque mot du vocabulaire pour la position **suivante**. Pour générer, seule la dernière position nous intéresse.

## Fonction `forward` adaptée aux besoins du décodage

Nous utiliserons la fonction `forward` suivante, qui rend un résultat adapté au décodage : seule la dernière position nous intéresse (la prédiction du mot suivant).

Elle utilise le cache `past_key_values` : il évite de recalculer, à chaque nouveau mot, les clés et valeurs d'attention de tous les mots précédents. Quand le cache est donné, il ne faut passer au modèle **que les nouveaux tokens**.

In [ ]:
@torch.no_grad()
def forward(tokens: torch.Tensor, past=None) -> tuple[torch.Tensor, object]:
  output = model(tokens, past_key_values=past, use_cache=True)
  return output.logits[:, -1, :], output.past_key_values

Il faut donc utiliser au choix :

- `forward(all_tokens)`
- `forward(last_token, past)`

où `all_tokens` serait par exemple une séquence de 7 tokens de forme `(1, 7)` là où `last_token` serait de forme `(1, 1)`.

*Pourquoi est-il souvent utile de conserver des dimensions à `1` dans les réseaux de neurones ?*

*Votre réponse à compléter ici.*

### Solution

Ces dimensions permettent de calculer en parallèle des résultats pour plusieurs exemples (batching) : le même code fonctionne pour un batch de 1 comme de 64.

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Encodage de tout ce qui a été produit jusqu'ici (entrées et sorties précédentes)
2. Choix de l'indice du mot suivant
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (nous utiliserons seulement le nombre de mots produits dans ce TP)

L'étape 2 est celle où tout se joue. Le code suivant implémente tout le reste, afin de pouvoir nous concentrer sur l'étape 2 : chaque méthode de décodage sera une fonction qui reçoit les logits du dernier mot (forme `(1, taille du vocabulaire)`) et renvoie l'indice du mot choisi (un tenseur de forme `()`).

In [ ]:
def decoding_loop(step_function):

  @functools.wraps(step_function)
  def wrapper(prompt: str, length: int, *step_args, **step_kwargs) -> str:
    decoded = [encode(prompt)]
    past = None
    for _ in tqdm.auto.trange(length, desc="Mots", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded.append(index.reshape(1, 1))
    return decode(torch.cat(decoded, dim=1))

  return wrapper

## Décodage glouton (*greedy*)

Le décodage glouton est la méthode la plus simple pour décoder du texte depuis un modèle appris : à chaque étape, on prend le mot de plus forte probabilité.

Dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$$

le décodage glouton choisit $\text{Mot}_i$ pour maximiser $P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un mot moins probable pour ensuite atteindre des mots très probables.

L'exemple suivant est tiré de l'article de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

Après « The », on choisit « nice » parce que c'est le mot le plus probable, alors que le texte complet « The nice woman » a une probabilité (0,20) inférieure à celle qu'on aurait obtenue en choisissant « dog » (« The dog has », 0,36).

*Codez la fonction `greedy(logits: torch.Tensor) -> torch.Tensor` qui prend en entrée les logits du modèle et renvoie l'indice du mot choisi.*

*Elle utilisera [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html). La fonction `decoding_loop` attend un tenseur de forme `()`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor:
  return torch.argmax(logits, dim=-1)[0]


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))

## Test sur des inputs variés

*Codez une fonction `test_decoding(function, *args, **kwargs) -> None` qui prend en entrée une méthode de décodage et ses arguments, et affiche les résultats de cette méthode sur les inputs ci-dessous. Testez la méthode `greedy` avec cette fonction. Que constatez-vous ?*

In [ ]:
if lang == "fr":
  inputs = ["Je suis allé au",
            "Le train pour Lyon",
            "Comment vas-tu",
            "La recette de la tarte aux pommes commence par",
            "Ça va merci, tu devrais",
            "Le chat de la voisine",
            "En 2050, les voitures"]
else:
  inputs = ["I went to the",
            "The train to London",
            "How do you",
            "The apple pie recipe starts with",
            "I'm fine thank you, you should",
            "The neighbour's cat",
            "In 2050, cars"]

In [ ]:
# Votre code ici

### Solution

In [ ]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = [function(prompt, *args, **kwargs)
                 for prompt in tqdm.auto.tqdm(inputs, desc="Prompts", leave=False)]
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)


test_decoding(greedy, 50)

On constate que les réponses « bouclent » assez rapidement : elles répètent la même phrase. C'est une faiblesse très classique du décodage glouton.

## Échantillonnage (*sampling*)

Les résultats du décodage glouton bouclent rapidement et sont souvent très génériques. Les méthodes suivantes cherchent à corriger ces défauts.

La première consiste à **échantillonner** à partir des probabilités du modèle, plutôt que de toujours choisir le mot le plus probable.

*Reprenez la forme de la fonction `greedy`, mais au lieu de choisir l'élément maximal, calculez une distribution de probabilités avec [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html), puis utilisez [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) pour tirer le prochain mot selon cette distribution.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor:
  probabilities = torch.softmax(logits, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


test_decoding(sampling, 50)

## Température

Avant la softmax, on peut diviser les logits par une **température** $T$ :

- $T < 1$ accentue les écarts : le modèle choisit plus souvent les mots les plus probables (plus cohérent, moins varié) ;
- $T > 1$ les aplatit : les mots rares sont plus souvent tirés (plus varié, moins cohérent) ;
- $T \to 0$ revient au décodage glouton.

*Ajoutez un paramètre `temperature` à votre fonction d'échantillonnage dans une nouvelle fonction `temperature_sampling`, et comparez les générations pour $T = 0.5$ et $T = 1.5$.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor:
  probabilities = torch.softmax(logits / temperature, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


test_decoding(temperature_sampling, 50, temperature=0.5)
test_decoding(temperature_sampling, 50, temperature=1.5)

## Top-k sampling

Une variation de l'échantillonnage consiste à ne tirer que parmi les `k` mots les plus probables, pour éviter de générer un mot très improbable.

*Repartez de la fonction `sampling` pour coder la fonction `k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor` qui implémente cette amélioration avec [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html). Testez-la avec la fonction `test_decoding`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values, dim=-1)
  sampled = torch.multinomial(probabilities, num_samples=1)[0, 0]
  return indices[0, sampled]


test_decoding(k_sampling, 50, k=20)

## Top-p sampling

Une autre amélioration consiste à ne garder que les mots les plus probables dont la probabilité cumulée atteint `p` (le *noyau* de la distribution), et pas les suivants. Le nombre de mots candidats s'adapte alors à la confiance du modèle.

*Repartez de la fonction `k_sampling` et ajoutez-y le top-p sampling, dans la fonction `k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor`. Vous pourrez utiliser [`torch.cumsum`](https://docs.pytorch.org/docs/stable/generated/torch.cumsum.html). Testez-la avec la fonction `test_decoding`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values[0], dim=-1)
  # Nombre de mots nécessaires pour que la probabilité cumulée atteigne p
  n_kept = int((torch.cumsum(probabilities, dim=0) < p).sum()) + 1
  sampled = torch.multinomial(probabilities[:n_kept], num_samples=1)[0]
  return indices[0, sampled]


test_decoding(k_p_sampling, 100, k=20, p=0.85)

## Utilisation des fonctions de `transformers`

En pratique, pour générer du texte, on utilise la méthode `generate` des modèles de la librairie `transformers`, qui implémente toutes ces stratégies (et d'autres, comme la recherche en faisceau) :

In [ ]:
def transformers_generate(prompt: str, length: int, temperature: float,
                          top_k: int, top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(token_ids,
                             attention_mask=torch.ones_like(token_ids),
                             max_new_tokens=length,
                             do_sample=True,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             pad_token_id=tokenizer.eos_token_id)
  return decode(generated)


test_decoding(transformers_generate, length=100, temperature=1.0, top_k=30, top_p=0.95)